# Event Horizon AI: Building Process Documentation from Video with Gemini

### Turning Screen Recordings into Automation-Ready Documentation Using 6 Gemini API Features

---

**Author:** Krzysztof "Chris" Karaszewski  
**Competition:** Google - Gemini Long Context  
**Repository:** Event Horizon AI (Video to PDD)

---

This notebook demonstrates the core Gemini API capabilities that power **Event Horizon AI** - a production web application that transforms screen recordings into structured Process Design Documents (PDD) for RPA automation.

The full application is built with Next.js + Convex + React Flow, but here we isolate and showcase the **six Gemini features** that make it possible:

| # | Gemini Feature | What It Does in Our App |
|---|---|---|
| 1 | **Long Context Video Understanding** | Watches entire screen recordings and understands every action |
| 2 | **Structured Output (JSON Schema)** | Forces AI to return perfectly typed PDD documents |
| 3 | **Spatial Understanding (Bounding Boxes)** | Locates exact UI elements in screenshots |
| 4 | **Context Caching** | Caches video tokens for 75% cost reduction across agent iterations |
| 5 | **Function Calling (Tool Use)** | Powers the ReAct agent loop that incrementally builds documents |
| 6 | **Thinking Mode Control** | Disabled for spatial tasks, enabled for reasoning |

> **Architecture Note:** The production app uses a **ReAct (Reason + Act) agent loop** - the same pattern used by Gemini CLI - where the AI autonomously reasons, calls tools, observes results, and iterates until the document is complete.

## The Problem: Inadequate Process Documentation

Every RPA (Robotic Process Automation) project begins with **process documentation** - a detailed, step-by-step record of how a business process is performed on screen.

### The Current Pain

- A business analyst sits with a subject-matter expert, watches them work, and **manually writes a Process Design Document (PDD)**
- A 5-minute screen recording takes **2-4 hours** to document properly
- Analysts miss steps, misidentify UI elements, forget branching logic
- **60-70% of RPA project time** is spent on process discovery and documentation, not building automation

### What Companies Currently Use

| Tool | Limitation |
|---|---|
| Manual observation + Word/Excel | Slow, inconsistent, error-prone |
| Process mining (Celonis, UiPath) | Only captures system logs, not screen-level actions |
| Screen recording + manual annotation | Still requires hours of human review |
| OCR-based tools | Can't understand context, workflow, or decisions |

### Why Only Gemini Can Do This

| Requirement | Why Gemini |
|---|---|
| **Watch 5-30 min videos** | 1M+ token context window handles long recordings natively |
| **Understand UI actions** | Native video understanding - sees clicks, typing, navigation in context |
| **Return structured data** | JSON Schema structured output ensures valid, typed responses |
| **Locate UI elements** | Spatial understanding with bounding box detection (0-1000 coordinate system) |
| **Build documents iteratively** | Function calling enables ReAct agent loops |
| **Keep costs manageable** | Context caching reduces video re-processing costs by 75% |

## Solution Architecture: A ReAct Agent Powered by Gemini

Event Horizon AI uses **two analysis modes**, both powered by Gemini:

### Mode 1: Single-Pass Analysis (Structured Output)
```
Video -> Gemini (JSON Schema) -> Complete PDD in one call
```
Uses: Video Understanding + Structured Output

### Mode 2: ReAct Agent Loop (Production Architecture)
```
                    +---> REASON (Gemini analyzes video context)
                    |         |
                    |         v
  Video (cached) ---+     ACT (call read_pdd / write_pdd tools)
                    |         |
                    |         v
                    +--- OBSERVE (feed tool results back)
                    |         |
                    |         v
                    +--- REPEAT until document is complete
```
Uses: Context Caching + Function Calling + Video Understanding

This is the **same pattern used by Gemini CLI** - an autonomous agent that:
1. **Reasons** about what to do next (analyze a section of video, add steps, create flowchart)
2. **Acts** by calling tools (`write_pdd` to add data, `read_pdd` to review progress)
3. **Observes** the tool results and decides the next action
4. **Repeats** until the PDD is complete (up to 50 iterations)

### Post-Processing: Bounding Box Detection
```
Screenshots -> Gemini (Spatial Understanding) -> Bounding boxes for each UI element
```
Uses: Spatial Understanding + Thinking Mode Control

---
## Configuration

Install the Google GenAI SDK and set up the API key.

In [ ]:
# Install the Google GenAI SDK
!pip install -q google-genai Pillow requests

In [ ]:
import json
import time
import base64
import requests
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
from google import genai
from google.genai import types

# For Kaggle: use Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GEMINI_API_KEY = secrets.get_secret("GOOGLE_API_KEY")
except:
    import os
    GEMINI_API_KEY = os.environ.get("GOOGLE_API_KEY", "YOUR_API_KEY_HERE")

# Initialize the client
client = genai.Client(api_key=GEMINI_API_KEY)

print("Google GenAI SDK ready.")

---
## Feature 1: Long Context Video Understanding

Gemini's native video understanding is the foundation of everything. Unlike OCR or frame-by-frame approaches, Gemini **watches the video as a continuous sequence** and understands:

- What the user clicked and where
- What text was typed into which field
- Page navigations and application switches
- Decision points and branching logic
- Loading states and system responses

### How We Use It in the Production App

The video is uploaded to the Gemini File API, and then referenced in content generation calls. The same video URI is reused across all agent iterations via context caching.

```typescript
// From our production code: convex/geminiApi.ts
// Upload video buffer to Gemini File API
const uploadResult = await uploadBufferToGemini(
  apiKey, videoBuffer, "video/mp4", `video-${jobId}`
);

// Then reference the video in generation calls
const response = await generateContentWithGemini(
  apiKey,
  "gemini-2.5-flash",
  uploadResult.uri,       // Video URI from File API
  uploadResult.mimeType,  // "video/mp4"
  fullPrompt,             // System + User prompt
  { responseSchema }      // JSON Schema for structured output
);
```

In [ ]:
# ============================================================
# FEATURE 1: Upload Video to Gemini File API
# ============================================================
# In the production app, videos are uploaded from Convex storage
# to the Gemini File API. Here we demonstrate the same flow.

# For this demo, we use a sample video from the dataset
# In production: video comes from user upload -> Convex storage -> Gemini File API

import os
from pathlib import Path

# Check for video file in Kaggle dataset
video_path = None
dataset_dirs = [
    "/kaggle/input/rpa-credit-bank-business-process-videos",
    "/kaggle/input"
]

for d in dataset_dirs:
    if os.path.exists(d):
        for f in os.listdir(d):
            if f.endswith(('.mp4', '.webm', '.mov', '.avi')):
                video_path = os.path.join(d, f)
                break
    if video_path:
        break

if video_path:
    print(f"Found video: {video_path}")
    print(f"Size: {os.path.getsize(video_path) / 1024 / 1024:.1f} MB")

    # Upload to Gemini File API
    video_file = client.files.upload(
        file=video_path,
        config=types.UploadFileConfig(mime_type="video/mp4")
    )

    # Wait for processing
    print(f"Uploaded: {video_file.name} (state: {video_file.state})")
    while video_file.state == "PROCESSING":
        time.sleep(5)
        video_file = client.files.get(name=video_file.name)
        print(f"  Processing... (state: {video_file.state})")

    print(f"Video ready: {video_file.uri}")
else:
    print("No video file found. Attach an RPA process recording to try video analysis.")
    print("The notebook will still demonstrate all Gemini features with text examples.")

---
## Feature 2: Structured Output (JSON Schema)

This is arguably the most critical Gemini feature for our use case. Instead of parsing free-text AI output, we **force Gemini to return a valid JSON object matching our exact schema**.

The production app defines a comprehensive JSON Schema covering:
- Process metadata (name, description, applications used)
- Step array with 37 UI element types, 34 specific actions, 9 screen regions
- Flowchart structure (8 node types, edge connections with conditions)
- Data mapping with sensitivity flags
- Business rules and exception scenarios

### How It Works

```typescript
// From our production code: convex/geminiApi.ts
const requestBody = {
  contents: [{ role: "user", parts }],
  generationConfig: {
    responseMimeType: "application/json",  // Forces JSON output
    responseSchema: jsonSchema,              // Our PDD schema
    maxOutputTokens: 65536,
  }
};
```

The schema ensures every response contains valid, typed data that maps directly to our database tables - no parsing, no guessing, no malformed output.

In [ ]:
# ============================================================
# FEATURE 2: Structured Output with JSON Schema
# ============================================================
# This is how we force Gemini to return valid PDD documents.
# The schema below is a simplified version of our production schema.

# Simplified PDD JSON Schema (production version has 200+ lines)
PDD_SCHEMA = {
    "type": "object",
    "properties": {
        "processes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "process_id": {"type": "string"},
                    "process_name": {"type": "string"},
                    "process_description": {"type": "string"},
                    "is_main_process": {"type": "boolean"},
                    "applications": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": {"type": "string"},
                                "type": {"type": "string", "enum": ["web_application", "desktop_application", "mobile_application", "terminal", "other"]},
                                "version": {"type": "string"}
                            },
                            "required": ["name", "type"]
                        }
                    },
                    "business_rules_observed": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "exceptions_noted": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "steps": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "step_number": {"type": "integer"},
                                "timestamp": {"type": "string"},
                                "action_type": {"type": "string", "enum": ["ui_interaction", "navigation", "data_transfer", "explanation", "wait", "validation"]},
                                "specific_action": {"type": "string", "enum": ["click", "double_click", "right_click", "type", "select", "check", "uncheck", "drag_and_drop", "scroll", "hover", "navigate_to_url", "open_application", "close_application", "switch_tab", "switch_window", "go_back", "read", "copy", "paste", "download", "upload", "export", "import", "note", "decision", "business_rule", "exception", "wait_for_element", "wait_for_page", "wait_for_process", "wait_fixed_time", "verify_element", "verify_value", "verify_state"]},
                                "description": {"type": "string"},
                                "application": {"type": "string"},
                                "screen_name": {"type": "string"},
                                "ui_element": {
                                    "type": "object",
                                    "properties": {
                                        "element_name": {"type": "string"},
                                        "element_type": {"type": "string", "enum": ["button", "link", "text_field", "text_area", "dropdown", "combobox", "checkbox", "radio_button", "toggle", "slider", "date_picker", "time_picker", "file_upload", "menu", "menu_item", "tab", "table", "table_row", "table_cell", "tree_view", "tree_node", "list", "list_item", "card", "modal", "dialog", "tooltip", "notification", "icon", "image", "label", "heading", "paragraph", "breadcrumb", "pagination", "search_field", "other"]},
                                        "location_description": {"type": "string"},
                                        "screen_region": {"type": "string", "enum": ["top_left", "top_center", "top_right", "middle_left", "middle_center", "middle_right", "bottom_left", "bottom_center", "bottom_right", "full_screen"]}
                                    },
                                    "required": ["element_name", "element_type", "screen_region"]
                                },
                                "data_info": {
                                    "type": "object",
                                    "properties": {
                                        "value": {"type": "string"},
                                        "data_type": {"type": "string", "enum": ["text", "number", "date", "datetime", "currency", "percentage", "boolean", "email", "phone", "url", "file", "password", "other"]},
                                        "source": {"type": "string", "enum": ["user_input", "system_generated", "database", "external_api", "file_import", "calculation", "other"]},
                                        "is_sensitive": {"type": "boolean"}
                                    },
                                    "required": ["value", "data_type", "source", "is_sensitive"]
                                },
                                "flow_node_id": {"type": "string"}
                            },
                            "required": ["step_number", "timestamp", "action_type", "specific_action", "description", "flow_node_id"]
                        }
                    },
                    "flow": {
                        "type": "object",
                        "properties": {
                            "nodes": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "properties": {
                                        "node_id": {"type": "string"},
                                        "node_type": {"type": "string", "enum": ["start", "end", "action", "decision", "switch", "merge", "subprocess", "loop_back"]},
                                        "label": {"type": "string"},
                                        "step_number": {"type": "integer"},
                                        "condition": {"type": "string"}
                                    },
                                    "required": ["node_id", "node_type", "label"]
                                }
                            },
                            "edges": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "properties": {
                                        "edge_id": {"type": "string"},
                                        "from_node_id": {"type": "string"},
                                        "to_node_id": {"type": "string"},
                                        "label": {"type": "string"},
                                        "edge_type": {"type": "string", "enum": ["normal", "exception", "timeout", "loop"]}
                                    },
                                    "required": ["edge_id", "from_node_id", "to_node_id"]
                                }
                            }
                        },
                        "required": ["nodes", "edges"]
                    }
                },
                "required": ["process_id", "process_name", "process_description", "is_main_process", "steps", "flow"]
            }
        }
    },
    "required": ["processes"]
}

print("PDD Schema defined.")
print(f"Schema covers: {len(json.dumps(PDD_SCHEMA))} chars of type definitions")
print(f"Action types: 6 categories, 34 specific actions")
print(f"UI element types: 37 types")
print(f"Flowchart nodes: 8 types (start, end, action, decision, switch, merge, subprocess, loop_back)")

In [ ]:
# ============================================================
# System Prompt - The "Brain" of the Analysis
# ============================================================
# This is the actual system prompt from our production app.
# It instructs Gemini how to analyze videos for RPA documentation.

SYSTEM_PROMPT = """You are an expert RPA (Robotic Process Automation) Business Analyst specializing in creating Process Design Documents (PDD) from screen recordings. Your task is to analyze a video of a business process and extract detailed, structured documentation with FLOWCHART representation that can be used for automation development.

## Documentation Standards

### Step Descriptions
- Every step description MUST start with either "User" or "System" to indicate who performs the action
- Use present tense action verbs (clicks, types, selects, navigates, verifies)
- Reference UI element names in single quotes (e.g., "User clicks 'Submit' button")
- Be specific about what happens, not vague

### UI Element Location Guidelines
Use a 9-zone grid to describe element positions:
- top_left, top_center, top_right
- middle_left, middle_center, middle_right
- bottom_left, bottom_center, bottom_right
- full_screen (for modals, overlays)

### Timestamp Format
Use MM:SS.s format (e.g., "00:05.2" for 5.2 seconds, "01:30.0" for 1 minute 30 seconds)

### Sensitive Data Handling
- Passwords: Show as "[MASKED]"
- Personal Identifiable Information (PII): Show as "[PII MASKED]"
- Credit card numbers: Show as "[CC MASKED]"
- SSN/Government IDs: Show as "[ID MASKED]"
- Always set is_sensitive: true for these data types

### Generic Data & Variable Standardization (CRITICAL)
- Replace specific business data with generic placeholders: "[Customer Name]", "[Invoice Number]", etc.
- Define and use key variables with {{VariableName}} format consistently

## FLOWCHART STRUCTURE
Produce a flowchart with NODES and EDGES:
- Node types: start, end, action, decision, switch, merge, subprocess, loop_back
- Edge types: normal, exception, timeout, loop
- Decision nodes have exactly TWO outgoing edges (Yes/No)
- Switch nodes have 3+ outgoing edges

Be thorough - capture EVERY action and create appropriate decision nodes for any branching logic observed."""

print(f"System prompt: {len(SYSTEM_PROMPT)} characters")
print("Covers: step descriptions, UI location grid, timestamps, data masking, flowchart structure")

In [ ]:
# ============================================================
# Run Video Analysis with Structured Output
# ============================================================
# This demonstrates the single-pass analysis mode.
# The video + system prompt + JSON schema go in -> structured PDD comes out.

if video_path and video_file:
    USER_PROMPT = """Analyze this screen recording video and generate a complete Process Design Document (PDD) 
    with FLOWCHART structure in JSON format. Document every user action, system response, and decision point."""

    full_prompt = SYSTEM_PROMPT + "\n\n" + USER_PROMPT

    print("Analyzing video with Gemini (structured output mode)...")
    start_time = time.time()

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            types.Content(
                role="user",
                parts=[
                    types.Part.from_uri(file_uri=video_file.uri, mime_type="video/mp4"),
                    types.Part.from_text(text=full_prompt),
                ]
            )
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=PDD_SCHEMA,
            max_output_tokens=65536,
        )
    )

    elapsed = time.time() - start_time
    print(f"Analysis completed in {elapsed:.1f}s")

    # Parse the structured response
    pdd_result = json.loads(response.text)

    # Display results
    for proc in pdd_result.get("processes", []):
        print(f"\nProcess: {proc['process_name']}")
        print(f"  Description: {proc['process_description'][:100]}...")
        print(f"  Steps: {len(proc.get('steps', []))}")
        print(f"  Flow nodes: {len(proc.get('flow', {}).get('nodes', []))}")
        print(f"  Flow edges: {len(proc.get('flow', {}).get('edges', []))}")

        for step in proc.get('steps', [])[:5]:
            ui = step.get('ui_element', {})
            print(f"    Step {step['step_number']} [{step['timestamp']}]: {step['description'][:80]}")
            if ui:
                print(f"      -> {ui.get('element_type', '?')}: '{ui.get('element_name', '?')}' @ {ui.get('screen_region', '?')}")

    # Token usage
    if response.usage_metadata:
        print(f"\nToken usage:")
        print(f"  Input:  {response.usage_metadata.prompt_token_count:,}")
        print(f"  Output: {response.usage_metadata.candidates_token_count:,}")
        print(f"  Total:  {response.usage_metadata.total_token_count:,}")
else:
    print("Skipping video analysis (no video file). See Feature 3 for bounding box demo.")

---
## Feature 3: Spatial Understanding - Bounding Box Detection

After the PDD is generated, Event Horizon AI runs a **second pass** over each screenshot to locate the exact UI element that was interacted with.

This uses Gemini's **spatial understanding** capability:
- Send a screenshot + description of the target UI element
- Gemini returns `box_2d: [ymin, xmin, ymax, xmax]` coordinates normalized to a **0-1000 scale**
- These coordinates are drawn as bounding box overlays on the screenshots

### Key Design Decisions

1. **Images resized to max 640px** - Google's recommendation for optimal bounding box accuracy
2. **Temperature set to 0** - Deterministic output for consistent spatial detection
3. **Thinking mode DISABLED** (`thinkingBudget: 0`) - Thinking actually *hurts* spatial understanding performance
4. **One element per call** - Each screenshot is analyzed for one specific UI element, using context from the PDD step

### Production Code

```typescript
// From our production code: convex/boundingBoxes.ts
const result = await ai.models.generateContent({
  model: "gemini-2.5-flash",
  contents: [{
    role: "user",
    parts: [
      { inlineData: { mimeType: "image/png", data: base64Image } },
      { text: prompt }  // Targeted prompt with element details
    ]
  }],
  config: {
    temperature: 0,                        // Deterministic
    thinkingConfig: { thinkingBudget: 0 }  // Disabled for spatial tasks
  }
});
```

### The Prompt Template

Each call includes rich context about what to find:

```
TARGET ELEMENT TO FIND:
- Element name/label: "Submit Button"
- Element type: button
- Location description: Bottom right of the form
- Screen region: bottom_right
- User action context: User clicks 'Submit' button to save the form
```

In [ ]:
# ============================================================
# FEATURE 3: Bounding Box Detection
# ============================================================
# This demonstrates how we locate UI elements in screenshots.
# We create a sample form screenshot and ask Gemini to find elements.

def create_sample_form_screenshot():
    """Create a simple form screenshot for demonstration."""
    img = Image.new('RGB', (800, 600), '#f0f4f8')
    draw = ImageDraw.Draw(img)

    # Title bar
    draw.rectangle([0, 0, 800, 50], fill='#1e40af')
    draw.text((20, 15), "Credit Application Form - Bank Portal", fill='white')

    # Form fields
    y = 80
    fields = [
        ("Customer Name", "John Smith"),
        ("Email Address", "john.smith@email.com"),
        ("Credit Amount", "$15,000.00"),
        ("Account Number", "****-****-1234"),
    ]

    for label, value in fields:
        draw.text((50, y), label, fill='#374151')
        draw.rectangle([50, y + 20, 400, y + 50], outline='#9ca3af', width=2)
        draw.rectangle([51, y + 21, 399, y + 49], fill='white')
        draw.text((60, y + 28), value, fill='#111827')
        y += 80

    # Submit button
    draw.rectangle([50, y + 10, 200, y + 50], fill='#2563eb')
    draw.text((90, y + 22), "Submit", fill='white')

    # Cancel button
    draw.rectangle([220, y + 10, 370, y + 50], fill='#dc2626')
    draw.text((260, y + 22), "Cancel", fill='white')

    return img

# Create the sample screenshot
sample_img = create_sample_form_screenshot()

# Convert to base64 for Gemini
buffer = BytesIO()
sample_img.save(buffer, format='PNG')
img_bytes = buffer.getvalue()
img_base64 = base64.b64encode(img_bytes).decode('utf-8')

print(f"Sample screenshot: {sample_img.size[0]}x{sample_img.size[1]} pixels")
print("Form contains: 4 text fields + Submit button + Cancel button")

In [ ]:
# ============================================================
# Detect UI Element Bounding Box
# ============================================================
# This is the exact same approach used in our production app.
# Key settings: temperature=0, thinking DISABLED.

BOUNDING_BOX_PROMPT = """You are a UI element detector. Find the EXACT bounding box of ONE specific UI element in this screenshot.

TARGET ELEMENT TO FIND:
- Element name/label: "Submit"
- Element type: button
- Location description: Below the form fields
- Screen region: bottom_left
- User action context: User clicks 'Submit' button to save the credit application

IMPORTANT INSTRUCTIONS:
1. Look for a visible label or text that says "Submit" or similar
2. The bounding box should cover the INTERACTIVE element itself (the button), NOT the label
3. The element type "button" helps identify - look for a clickable button

Return a JSON object:
{
  "label": "Submit",
  "box_2d": [ymin, xmin, ymax, xmax],
  "found": true
}

Where box_2d coordinates are normalized to 0-1000 scale (0=top/left, 1000=bottom/right).

If the element cannot be found, return:
{"label": "Submit", "box_2d": [0, 0, 0, 0], "found": false}

Return ONLY the JSON object, no other text."""

print("Detecting 'Submit' button bounding box...")
start_time = time.time()

bb_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_bytes(data=img_bytes, mime_type="image/png"),
                types.Part.from_text(text=BOUNDING_BOX_PROMPT),
            ]
        )
    ],
    config=types.GenerateContentConfig(
        temperature=0,                                     # Deterministic output
        thinking_config=types.ThinkingConfig(thinking_budget=0),  # DISABLED for spatial tasks
    )
)

elapsed = time.time() - start_time
print(f"Detection completed in {elapsed:.1f}s")

# Parse result
response_text = bb_response.text.strip()
if "```json" in response_text:
    response_text = response_text.split("```json")[1].split("```")[0].strip()
elif "```" in response_text:
    response_text = response_text.split("```")[1].split("```")[0].strip()

bb_result = json.loads(response_text)
print(f"\nResult: {json.dumps(bb_result, indent=2)}")
print(f"\nCoordinates (0-1000 scale): ymin={bb_result['box_2d'][0]}, xmin={bb_result['box_2d'][1]}, ymax={bb_result['box_2d'][2]}, xmax={bb_result['box_2d'][3]}")

In [ ]:
# ============================================================
# Draw Bounding Box Overlay
# ============================================================
# In the production app, this is handled by convex/boundingBoxOverlay.ts
# using Jimp. Here we use PIL to draw the same overlay.

def draw_bounding_box(image, box_2d, label, is_sensitive=False):
    """Draw a bounding box overlay on the image.
    
    Coordinates are in 0-1000 normalized scale.
    Green for normal elements, red for sensitive data.
    """
    img = image.copy()
    draw = ImageDraw.Draw(img)
    w, h = img.size
    
    # Convert from 0-1000 scale to pixel coordinates
    ymin, xmin, ymax, xmax = box_2d
    x1 = int(xmin / 1000 * w)
    y1 = int(ymin / 1000 * h)
    x2 = int(xmax / 1000 * w)
    y2 = int(ymax / 1000 * h)
    
    # Color: red for sensitive, green for normal
    color = '#dc2626' if is_sensitive else '#22c55e'
    
    # Draw rectangle (3px border)
    for i in range(3):
        draw.rectangle([x1-i, y1-i, x2+i, y2+i], outline=color)
    
    # Draw label background
    label_text = f"[SENSITIVE] {label}" if is_sensitive else label
    text_bbox = draw.textbbox((0, 0), label_text)
    text_w = text_bbox[2] - text_bbox[0] + 10
    text_h = text_bbox[3] - text_bbox[1] + 6
    draw.rectangle([x1, y1 - text_h - 2, x1 + text_w, y1 - 2], fill=color)
    draw.text((x1 + 5, y1 - text_h), label_text, fill='white')
    
    return img

if bb_result.get('found', False):
    # Draw the bounding box
    annotated = draw_bounding_box(sample_img, bb_result['box_2d'], bb_result['label'])
    
    # Save and display
    annotated.save("bounding_box_demo.png")
    print("Bounding box overlay saved to: bounding_box_demo.png")
    
    # Display inline
    from IPython.display import display
    display(annotated)
else:
    print("Element not found - no bounding box to draw.")

In [ ]:
# ============================================================
# Sensitive Data Detection with Bounding Boxes
# ============================================================
# The production app also detects sensitive data in screenshots.
# Uses the same spatial understanding but returns MULTIPLE boxes.

SENSITIVE_INFO_PROMPT = """You are a sensitive information detector for RPA documentation.

USER'S SENSITIVE INFO DEFINITION:
Detect any personal identifiable information (PII), account numbers, email addresses, 
financial amounts, and names that should be masked in documentation.

STEP CONTEXT:
User reviews the credit application form with customer details.

TASK: Find ALL instances of sensitive information in this screenshot.
For EACH instance found, return a bounding box with:
- The type of sensitive info detected
- The exact location as [ymin, xmin, ymax, xmax] normalized to 0-1000 scale
- A confidence score 0-1

Return a JSON object:
{
  "sensitive_boxes": [
    {
      "label": "Customer Name",
      "box_2d": [100, 200, 150, 400],
      "found": true,
      "confidence": 0.95
    }
  ]
}

If no sensitive information found, return:
{"sensitive_boxes": []}

Return ONLY the JSON object, no other text."""

print("Detecting sensitive information in screenshot...")
start_time = time.time()

sensitive_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_bytes(data=img_bytes, mime_type="image/png"),
                types.Part.from_text(text=SENSITIVE_INFO_PROMPT),
            ]
        )
    ],
    config=types.GenerateContentConfig(
        temperature=0,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )
)

elapsed = time.time() - start_time
print(f"Detection completed in {elapsed:.1f}s")

# Parse result
sens_text = sensitive_response.text.strip()
if "```json" in sens_text:
    sens_text = sens_text.split("```json")[1].split("```")[0].strip()
elif "```" in sens_text:
    sens_text = sens_text.split("```")[1].split("```")[0].strip()

sensitive_result = json.loads(sens_text)
print(f"\nFound {len(sensitive_result.get('sensitive_boxes', []))} sensitive items:")

# Draw all sensitive boxes on the image
annotated_sensitive = sample_img.copy()
for box in sensitive_result.get('sensitive_boxes', []):
    if box.get('found', True):
        annotated_sensitive = draw_bounding_box(
            annotated_sensitive, box['box_2d'], box['label'], is_sensitive=True
        )
        conf = box.get('confidence', 'N/A')
        print(f"  - {box['label']}: {box['box_2d']} (confidence: {conf})")

annotated_sensitive.save("sensitive_info_demo.png")
print("\nSensitive info overlay saved to: sensitive_info_demo.png")

from IPython.display import display
display(annotated_sensitive)

---
## Feature 4: Context Caching (75% Cost Reduction)

Video tokens are expensive. A 5-minute screen recording can consume **100K-500K tokens**. In the ReAct agent loop, the same video is referenced in every iteration (up to 50 iterations). Without caching, this would cost 50x the video tokens.

**Context Caching** solves this:

```
CACHED (once):                 SENT EACH ITERATION:
+---------------------------+  +------------------+
| Video file (100K+ tokens) |  | Conversation     |
| System prompt (10K)       |  | history          |
| Tool declarations         |  | (500-2K tokens)  |
| Tool config               |  +------------------+
+---------------------------+
```

### Cost Comparison (Gemini 2.5 Flash)

| Token Type | Standard Price | Cached Price | Savings |
|---|---|---|---|
| Input tokens | $0.15 / 1M | $0.0375 / 1M | **75%** |
| Output tokens | $3.50 / 1M | $3.50 / 1M | 0% |

For a typical 50-iteration analysis with 200K cached tokens:
- **Without caching:** 50 x 200K x $0.15/1M = **$1.50** in input tokens alone
- **With caching:** 1 x 200K x $0.0375/1M + 50 x 2K x $0.15/1M = **$0.02**

### Production Implementation

```typescript
// From our production code: convex/geminiApi.ts

// 1. Create cache with video + system prompt + tools
const { cacheName } = await createGeminiCache(
  apiKey, model,
  fileUri, fileMimeType,  // Video (the expensive part)
  systemPrompt,           // Full PDD instructions
  toolDeclarations,       // read_pdd + write_pdd tools
  ttlSeconds: 300         // 5-minute TTL
);

// 2. Use cache in every agent iteration (only send new messages)
const response = await generateContentWithCache(
  apiKey, model,
  cacheName,    // Reference cached video + prompt + tools
  messages       // Only the conversation history
);

// 3. Refresh TTL every 20 iterations
if (iteration % 20 === 0) {
  await refreshGeminiCacheTTL(apiKey, cacheName, 300);
}

// 4. Delete cache when done
await deleteGeminiCache(apiKey, cacheName);
```

In [ ]:
# ============================================================
# FEATURE 4: Context Caching Demo
# ============================================================
# Demonstrating the caching API that powers the ReAct agent loop.
# In production this caches the video + system prompt + tool declarations.

CACHE_SYSTEM_PROMPT = SYSTEM_PROMPT + "\n\nYou have access to tools: read_pdd (review document) and write_pdd (build document incrementally)."

if video_path and video_file:
    print("Creating context cache with video + system prompt...")

    cache = client.caches.create(
        model="gemini-2.5-flash",
        config=types.CreateCachedContentConfig(
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(file_uri=video_file.uri, mime_type="video/mp4"),
                    ]
                )
            ],
            system_instruction=CACHE_SYSTEM_PROMPT,
            ttl="300s",  # 5 minutes
        )
    )

    print(f"Cache created: {cache.name}")
    print(f"Cached token count: {cache.usage_metadata.total_token_count:,}")
    print(f"Expires: {cache.expire_time}")

    # Now generate using the cache (only send conversation, not video)
    print("\nGenerating with cache (cheap - no video re-processing)...")
    start_time = time.time()

    cached_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=["List the first 3 steps of the process you see in the video."],
        config=types.GenerateContentConfig(
            cached_content=cache.name,
        )
    )

    elapsed = time.time() - start_time
    print(f"Response in {elapsed:.1f}s:")
    print(cached_response.text[:500])

    if cached_response.usage_metadata:
        meta = cached_response.usage_metadata
        cached_count = getattr(meta, 'cached_content_token_count', 0) or 0
        total = meta.total_token_count
        print(f"\nTokens - Total: {total:,}, Cached: {cached_count:,}")
        if cached_count:
            savings = (cached_count / total) * 75  # 75% savings on cached tokens
            print(f"Cost savings: ~{savings:.0f}% of input tokens were cached")

    # Cleanup
    client.caches.delete(name=cache.name)
    print("\nCache deleted.")
else:
    print("Skipping cache demo (no video file). See architecture diagrams above.")
    print("\nIn the production app, caching saves 75% on video token costs across")
    print("the ~50 iterations of the ReAct agent loop.")

---
## Feature 5: Function Calling (The ReAct Agent's Hands)

This is where Event Horizon AI's architecture truly mirrors **Gemini CLI**. Instead of asking the AI to return a complete document in one shot, we give it **tools** and let it build the document incrementally.

### The Two Tools

| Tool | Purpose | Operations |
|---|---|---|
| `read_pdd` | Review current document state | `full`, `processes`, `steps`, `flow`, `metadata` |
| `write_pdd` | Build the document incrementally | `add_process`, `add_steps`, `set_flow`, `update_metadata`, `update_process` |

### The ReAct Loop in Action

```
Iteration 1:  Gemini reasons: "I see a login form. Let me create the process."
              -> calls write_pdd(operation="add_process", data={...})
              -> receives: {success: true, processId: "abc123"}

Iteration 2:  Gemini reasons: "Now I see the user typing credentials."
              -> calls write_pdd(operation="add_steps", data={steps: [...]})
              -> receives: {success: true, stepsAdded: 5}

Iteration 3:  Gemini reasons: "Let me check my progress so far."
              -> calls read_pdd(section="steps", process_id="abc123")
              -> receives: {steps: [{...}, {...}, ...]}

Iteration 4:  Gemini reasons: "I see a decision point - login success/failure."
              -> calls write_pdd(operation="set_flow", data={nodes: [...], edges: [...]})
              -> receives: {success: true}

...continues for up to 50 iterations...

Iteration N:  Gemini reasons: "The document is complete."
              -> returns text summary (no tool call = loop ends)
```

### Why This is Better Than Single-Pass

1. **Self-review**: The agent reads back what it wrote and catches errors
2. **Incremental building**: Complex processes are built piece by piece
3. **No output token limits**: Each tool call's data is small, avoiding truncation
4. **Persistent state**: Data is saved to the database after each tool call - crash-safe
5. **Pause/Resume**: User can pause and the agent picks up where it left off

### Production Code: Tool Declarations

```typescript
// From our production code: convex/agentTools.ts
export function getToolDeclarations(): GeminiToolDeclaration[] {
  return [
    {
      name: "read_pdd",
      description: "Read the current state of the PDD document being built.",
      parameters: {
        type: "object",
        properties: {
          section: {
            type: "string",
            enum: ["full", "processes", "steps", "flow", "metadata"]
          },
          process_id: { type: "string" }
        },
        required: ["section"]
      }
    },
    {
      name: "write_pdd",
      description: "Write or update a section of the PDD document.",
      parameters: {
        type: "object",
        properties: {
          operation: {
            type: "string",
            enum: ["set_processes", "add_process", "update_process",
                   "add_steps", "set_flow", "update_metadata"]
          },
          process_id: { type: "string" },
          data: { type: "object" }
        },
        required: ["operation", "data"]
      }
    }
  ];
}
```

In [ ]:
# ============================================================
# FEATURE 5: Function Calling Demo - ReAct Agent Loop
# ============================================================
# This simulates the production ReAct loop locally.
# In production, tools read/write to the Convex database.
# Here, tools read/write to a local dictionary.

# Local document store (simulates Convex database)
pdd_store = {
    "processes": [],
    "steps": {},
    "flows": {},
}

# Tool declarations (same as production)
TOOL_DECLARATIONS = [
    {
        "name": "read_pdd",
        "description": "Read the current state of the PDD document being built. Use this to check progress and review your work.",
        "parameters": {
            "type": "object",
            "properties": {
                "section": {
                    "type": "string",
                    "enum": ["full", "processes", "steps", "flow"],
                    "description": "Which section to read."
                },
                "process_id": {
                    "type": "string",
                    "description": "Required for steps/flow sections."
                }
            },
            "required": ["section"]
        }
    },
    {
        "name": "write_pdd",
        "description": "Write or update a section of the PDD document incrementally.",
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add_process", "add_steps", "set_flow", "update_metadata"],
                    "description": "The write operation."
                },
                "process_id": {"type": "string"},
                "data": {"type": "object", "description": "Data to write."}
            },
            "required": ["operation", "data"]
        }
    }
]

def execute_tool(tool_name, args):
    """Execute a tool call against the local PDD store."""
    if tool_name == "read_pdd":
        section = args.get("section", "full")
        pid = args.get("process_id")
        if section == "full":
            return {"success": True, "data": pdd_store}
        elif section == "processes":
            return {"success": True, "data": pdd_store["processes"]}
        elif section == "steps" and pid:
            return {"success": True, "data": pdd_store["steps"].get(pid, [])}
        elif section == "flow" and pid:
            return {"success": True, "data": pdd_store["flows"].get(pid, {})}
        return {"success": True, "data": pdd_store}

    elif tool_name == "write_pdd":
        op = args.get("operation")
        data = args.get("data", {})
        pid = args.get("process_id") or data.get("process_id", "proc_1")

        if op == "add_process":
            pdd_store["processes"].append(data)
            pdd_store["steps"][pid] = []
            pdd_store["flows"][pid] = {"nodes": [], "edges": []}
            return {"success": True, "process_id": pid}
        elif op == "add_steps":
            steps = data.get("steps", [])
            if pid not in pdd_store["steps"]:
                pdd_store["steps"][pid] = []
            pdd_store["steps"][pid].extend(steps)
            return {"success": True, "steps_added": len(steps)}
        elif op == "set_flow":
            pdd_store["flows"][pid] = data
            return {"success": True}
        elif op == "update_metadata":
            for proc in pdd_store["processes"]:
                if proc.get("process_id") == pid:
                    proc.update(data)
            return {"success": True}

    return {"success": False, "error": f"Unknown tool: {tool_name}"}

print("Tool functions defined. Ready for ReAct loop.")
print(f"Tools: {[t['name'] for t in TOOL_DECLARATIONS]}")

In [ ]:
# ============================================================
# Run the ReAct Agent Loop
# ============================================================
# This is a simplified version of convex/agentLoop.ts.
# Same pattern: Reason -> Act (call tools) -> Observe -> Repeat

AGENT_PROMPT = """You are building a Process Design Document (PDD) for an RPA automation project.

Create a sample PDD for a "Bank Login Process" with these steps:
1. User opens the bank website
2. User enters username
3. User enters password
4. User clicks Login button
5. System validates credentials (decision: success or failure)
6. If success: System displays dashboard
7. If failure: System shows error message

Use write_pdd to build the document step by step:
1. First, create the process with add_process
2. Then, add the steps with add_steps
3. Then, create the flowchart with set_flow
4. Finally, use read_pdd to verify the result

Build it incrementally - don't try to do everything in one call."""

# Conversation history (multi-turn)
messages = [{"role": "user", "parts": [{"text": AGENT_PROMPT}]}]

MAX_ITERATIONS = 10
total_tool_calls = 0

print("Starting ReAct Agent Loop...")
print("=" * 60)

for iteration in range(1, MAX_ITERATIONS + 1):
    print(f"\n--- Iteration {iteration} ---")

    # REASON: Call Gemini with conversation + tools
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=messages,
        config=types.GenerateContentConfig(
            tools=[types.Tool(function_declarations=[
                types.FunctionDeclaration(**td) for td in TOOL_DECLARATIONS
            ])],
            tool_config=types.ToolConfig(
                function_calling_config=types.FunctionCallingConfig(mode="AUTO")
            ),
        )
    )

    # Extract parts from response
    response_parts = response.candidates[0].content.parts

    # Check for thinking/reasoning text
    for part in response_parts:
        if part.text:
            print(f"  THINK: {part.text[:150]}{'...' if len(part.text) > 150 else ''}")

    # Check for function calls
    function_calls = [p for p in response_parts if p.function_call]

    if not function_calls:
        # No tool calls = agent is done
        print("  Agent finished (no more tool calls).")
        # Add model response to conversation
        messages.append({"role": "model", "parts": [{"text": part.text} for part in response_parts if part.text]})
        break

    # ACT: Execute each function call
    model_parts = []
    function_responses = []

    for part in response_parts:
        if part.function_call:
            fc = part.function_call
            tool_name = fc.name
            tool_args = dict(fc.args) if fc.args else {}

            print(f"  ACT:  {tool_name}({json.dumps(tool_args)[:100]})")

            # Execute the tool
            result = execute_tool(tool_name, tool_args)
            total_tool_calls += 1

            result_preview = json.dumps(result)[:100]
            print(f"  OBSERVE: {result_preview}")

            model_parts.append(types.Part.from_function_call(
                name=tool_name, args=tool_args
            ))
            function_responses.append(types.Part.from_function_response(
                name=tool_name, response=result
            ))
        elif part.text:
            model_parts.append(types.Part.from_text(text=part.text))

    # Add model response + function results to conversation
    messages.append({"role": "model", "parts": model_parts})
    messages.append({"role": "user", "parts": function_responses})

print("\n" + "=" * 60)
print(f"ReAct loop completed: {iteration} iterations, {total_tool_calls} tool calls")
print(f"\nFinal PDD State:")
print(f"  Processes: {len(pdd_store['processes'])}")
for pid, steps in pdd_store['steps'].items():
    print(f"  Steps ({pid}): {len(steps)}")
for pid, flow in pdd_store['flows'].items():
    print(f"  Flow nodes ({pid}): {len(flow.get('nodes', []))}")
    print(f"  Flow edges ({pid}): {len(flow.get('edges', []))}")

---
## Feature 6: Thinking Mode Control

Gemini 2.5 Flash supports **thinking mode** where the model reasons step-by-step before responding. Event Horizon AI strategically controls this:

| Task | Thinking Mode | Why |
|---|---|---|
| Video analysis (ReAct agent) | **Enabled** (default) | Complex reasoning about process logic, decisions, and flowcharts |
| Bounding box detection | **Disabled** (`thinkingBudget: 0`) | Spatial tasks perform *worse* with thinking - direct visual perception is more accurate |
| Sensitive data detection | **Disabled** (`thinkingBudget: 0`) | Same as bounding boxes - visual pattern matching, not reasoning |

This is a key insight from production: **thinking helps for logical analysis but hurts for spatial understanding**.

```typescript
// Spatial task: DISABLE thinking
config: {
  temperature: 0,
  thinkingConfig: { thinkingBudget: 0 }
}

// Reasoning task: ENABLE thinking (default)
// Just don't set thinkingConfig - model uses default budget
```

---
## Full Architecture: How It All Fits Together

```
                         EVENT HORIZON AI
                    Production Architecture

  +--------------------------------------------------+
  |                  NEXT.JS FRONTEND                 |
  |  Upload Page | Process Viewer | Flowchart Editor  |
  |  Agent Panel (live events) | Export (JSON/DOCX)   |
  +--------------------------------------------------+
              |           |            |
              v           v            v
  +--------------------------------------------------+
  |               CONVEX BACKEND                      |
  |                                                   |
  |  +--------------------------------------------+   |
  |  |        ReAct Agent Loop (agentLoop.ts)     |   |
  |  |                                            |   |
  |  |  while (iteration < maxIterations):        |   |
  |  |    1. REASON: Gemini analyzes video        |   |
  |  |    2. ACT: call read_pdd / write_pdd       |   |
  |  |    3. OBSERVE: feed results back           |   |
  |  |    4. EMIT: stream events to UI            |   |
  |  |    5. CHECK: user stop/pause requested?    |   |
  |  |                                            |   |
  |  +--------------------------------------------+   |
  |        |                    |                     |
  |        v                    v                     |
  |  +-----------+    +------------------+            |
  |  | Gemini API |   | Convex Database  |            |
  |  |            |   |                  |            |
  |  | - Cache    |   | - processes      |            |
  |  | - Generate |   | - steps          |            |
  |  | - Tools    |   | - processFlows   |            |
  |  +-----------+    | - agentSessions  |            |
  |                   | - agentEvents    |            |
  |                   +------------------+            |
  +--------------------------------------------------+
              |
              v
  +--------------------------------------------------+
  |          POST-PROCESSING (per screenshot)         |
  |                                                   |
  |  1. Extract screenshots (FFmpeg at timestamps)    |
  |  2. Bounding box detection (Gemini spatial)       |
  |     - temperature: 0, thinking: DISABLED          |
  |     - coordinates: 0-1000 normalized scale        |
  |  3. Sensitive info detection (same config)        |
  |     - Multiple boxes per screenshot               |
  |     - Red overlay for sensitive, green for normal |
  |  4. Overlay generation (Jimp image processing)    |
  +--------------------------------------------------+
```

### Gemini Features Used at Each Stage

| Stage | Features | Model |
|---|---|---|
| Video Upload | File API | - |
| Agent Analysis | Context Caching + Function Calling + Video Understanding | gemini-2.5-flash |
| Single-Pass Analysis | Structured Output (JSON Schema) + Video Understanding | configurable |
| Bounding Box Detection | Spatial Understanding + Thinking Control (disabled) | gemini-2.5-flash |
| Sensitive Info Detection | Spatial Understanding + Thinking Control (disabled) | gemini-2.5-flash |

### How This Mirrors Gemini CLI

| Gemini CLI | Event Horizon AI |
|---|---|
| Reads/writes files on disk | Reads/writes to Convex database |
| Runs shell commands | Calls read_pdd / write_pdd |
| Multi-turn conversation | Same - conversation persisted across iterations |
| Context caching for large codebases | Context caching for video files |
| Autonomous reasoning loop | Same ReAct pattern |
| Terminal UI with live output | Web UI with real-time event streaming |

---
## Cost Analysis: Caching vs No Caching

Real production numbers from analyzing a 3-minute screen recording:

In [ ]:
# ============================================================
# Cost Visualization: With and Without Caching
# ============================================================
# These are the actual cost rates from our production config
# (convex/agentLoop.ts AGENT_CONFIG)

# Pricing (Gemini 2.5 Flash)
INPUT_PER_1M = 0.15
OUTPUT_PER_1M = 3.50
CACHED_INPUT_PER_1M = 0.0375
CACHE_STORAGE_PER_1M_PER_HOUR = 1.00

# Typical analysis parameters
video_tokens = 250_000     # 3-minute video
system_prompt_tokens = 15_000
cached_tokens = video_tokens + system_prompt_tokens
per_iteration_input = 1_500   # Conversation history
per_iteration_output = 4_000  # Reasoning + tool calls
iterations = 35               # Typical for a medium process
analysis_duration_hours = 0.1  # ~6 minutes

# WITHOUT caching: full video sent every iteration
no_cache_input = (cached_tokens + per_iteration_input) * iterations
no_cache_output = per_iteration_output * iterations
no_cache_cost = (no_cache_input / 1_000_000 * INPUT_PER_1M +
                 no_cache_output / 1_000_000 * OUTPUT_PER_1M)

# WITH caching: video cached, only conversation sent per iteration
cache_input = per_iteration_input * iterations  # Only new tokens
cache_cached = cached_tokens * iterations       # Read from cache (75% cheaper)
cache_output = per_iteration_output * iterations
cache_storage = cached_tokens / 1_000_000 * CACHE_STORAGE_PER_1M_PER_HOUR * analysis_duration_hours
with_cache_cost = (cache_input / 1_000_000 * INPUT_PER_1M +
                   cache_cached / 1_000_000 * CACHED_INPUT_PER_1M +
                   cache_output / 1_000_000 * OUTPUT_PER_1M +
                   cache_storage)

savings = (1 - with_cache_cost / no_cache_cost) * 100

print("Cost Analysis: 3-minute video, 35 agent iterations")
print("=" * 55)
print(f"\nWITHOUT Context Caching:")
print(f"  Input tokens:  {no_cache_input:>12,} x ${INPUT_PER_1M}/1M")
print(f"  Output tokens: {no_cache_output:>12,} x ${OUTPUT_PER_1M}/1M")
print(f"  Total cost:    ${no_cache_cost:>11.4f}")
print(f"\nWITH Context Caching:")
print(f"  New input:     {cache_input:>12,} x ${INPUT_PER_1M}/1M")
print(f"  Cached input:  {cache_cached:>12,} x ${CACHED_INPUT_PER_1M}/1M (75% off)")
print(f"  Output tokens: {cache_output:>12,} x ${OUTPUT_PER_1M}/1M")
print(f"  Cache storage: {cached_tokens:>12,} x ${CACHE_STORAGE_PER_1M_PER_HOUR}/1M/hr")
print(f"  Total cost:    ${with_cache_cost:>11.4f}")
print(f"\nSavings: {savings:.0f}% (${no_cache_cost - with_cache_cost:.4f} saved per analysis)")
print(f"\nAt 100 analyses/month: ${(no_cache_cost - with_cache_cost) * 100:.2f}/month saved")

---
## Using the Video Recording as Documentation

A unique feature of Event Horizon AI is that the **original video becomes part of the documentation**. Each PDD step includes:

- **Timestamp** (MM:SS.s) linking back to the exact moment in the video
- **Auto-extracted screenshot** at that timestamp (via FFmpeg)
- **Bounding box overlay** showing exactly which UI element was interacted with
- **Sensitive data masks** highlighting any PII, passwords, or credentials

This means an RPA developer can:
1. Read the step description
2. See the annotated screenshot with the highlighted element
3. Click the timestamp to jump to that exact moment in the video
4. See the XPath, CSS selectors, and accessibility labels for automation

The video isn't just an input - it becomes a **living reference** embedded in the PDD.

---
## Summary

Event Horizon AI demonstrates how **six Gemini API features** combine to solve a real enterprise problem:

| Feature | Impact |
|---|---|
| **Long Context Video Understanding** | Watches entire recordings and understands every click, keystroke, and navigation |
| **Structured Output (JSON Schema)** | Guarantees valid, typed documents - no parsing failures, ever |
| **Spatial Understanding (Bounding Boxes)** | Locates exact UI elements in screenshots with 0-1000 normalized coordinates |
| **Context Caching** | 75% cost reduction - makes the ReAct agent loop economically viable |
| **Function Calling** | Powers the autonomous ReAct agent that builds documents incrementally |
| **Thinking Mode Control** | Enabled for reasoning, disabled for spatial tasks - best of both worlds |

### The ReAct Agent Architecture

The production app's **ReAct loop** mirrors the architecture of **Gemini CLI**:
- Autonomous reasoning with tool use
- Multi-turn conversation with persistent state
- Context caching for large inputs (video instead of codebase)
- Real-time event streaming to the UI
- Pause/resume with session persistence
- Cost tracking and token accounting

### The Result

| Metric | Manual Process | Event Horizon AI |
|---|---|---|
| Documentation time | 2-4 hours | 5-15 minutes |
| Consistency | Varies by analyst | Standardized every time |
| UI element detail | Basic descriptions | XPath, selectors, bounding boxes |
| Flowchart | Separate Visio work | Auto-generated |
| Sensitive data | Easy to miss | Auto-detected and masked |
| Cost per analysis | $80-150 (analyst time) | ~$0.50 (API costs) |

**What used to take a skilled analyst half a day now takes a screen recording and a few minutes of AI processing.**

---

*Built with Google Gemini 2.5 Flash, Next.js 16, Convex, and React Flow.*  
*Apache 2.0 License*